In [2]:
from pathlib import Path
import xarray as xr
import numpy as np

In [ ]:
mesh_path = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
W_path = "/work/bk1450/b383184/Amazon/Mercator/data/variables_c/W_1993-01c.nc"
SSH_path = "/work/bk1450/b383184/Amazon/Mercator/data/variables_c/SSH_1993-01c.nc"

In [ ]:
ds_mesh = xr.open_dataset(mesh_path, chunks={})
ds_mesh = ds_mesh.assign_coords(x=np.arange(ds_mesh.sizes["x"]))
ds_mesh = ds_mesh.assign_coords(y=np.arange(ds_mesh.sizes["y"]))
ds_mesh = ds_mesh.assign_coords(z=np.arange(ds_mesh.sizes["z"]))
ds_mesh = ds_mesh.squeeze()
ds_mesh

In [ ]:
#construct W depth for each water filled cell
e3t_full = xr.where(
    (ds_mesh.z + 1) <= (ds_mesh.mbathy - 1),  # wet cells above bottom cells
    ds_mesh.e3t_0,  # fill with basin wide e3t for level,
    ds_mesh.e3t_ps,  # add partial cell height otherwise
).where((ds_mesh.z + 1) <= ds_mesh.mbathy)  # remove all non-wet cells below
e3t_full

In [ ]:
depthw_ps = e3t_full.cumsum("z").where((ds_mesh.z + 1) <= ds_mesh.mbathy)
depthw_ps = depthw_ps.shift(z=1).fillna(0.0)
depthw_ps = depthw_ps.where(ds_mesh.z <= ds_mesh.mbathy)
depthw_ps = depthw_ps.rename({"z": "depthw"}).drop("depthw")

In [ ]:
H_bottom = e3t_full.sum("z")

In [ ]:
#Load W file
ds_W = xr.open_dataset(W_path, chunks={"time_counter": 1})
ds_W = ds_W.assign_coords(x=np.arange(ds_W.sizes["x"]))
ds_W = ds_W.assign_coords(y=np.arange(ds_W.sizes["y"]))
ds_W = ds_W.assign_coords(z=-depthw_ps, H=H_bottom)

In [ ]:
#Load SSH file
ds_SSH = xr.open_dataset(SSH_path, chunks={"time_counter": 1})
ds_SSH = ds_SSH.assign_coords(x=np.arange(ds_SSH.sizes["x"]))
ds_SSH = ds_SSH.assign_coords(y=np.arange(ds_SSH.sizes["y"]))

## Correcting W for fixed sea surface:

$z$ is positive upward, $H$ is positive, and $\eta$ is positive upward

$$W_{merc}(z) = W_\eta(z) + W_{fixed}(z)$$

$$W_\eta(z) = \frac{z+H}{\eta+H} \frac{\partial \eta}{\partial t}$$



In [ ]:
deta_dt = ds_W.vovecrtz.isel(depthw=0, drop=True)

In [ ]:
eta = ds_SSH.sossheig

In [ ]:
W_merc = ds_W.vovecrtz.rename("W_merc").fillna(0.0)

In [ ]:
W_eta = (ds_W.z + ds_W.H) / (eta + ds_W.H) * deta_dt
W_eta = W_eta.rename("W_eta")
W_eta = W_eta.assign_coords(z=ds_W.z)

In [ ]:
W_fixed = W_merc - W_eta
W_fixed = W_fixed.rename("W_fixed")

In [3]:
## old coords
x = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/U_1993-01c.nc', chunks={}).x
y = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/U_1993-01c.nc', chunks={}).y

In [ ]:
#re-define the x and y to match with the U and V
W_fixed = W_fixed.assign_coords(x=x)
W_fixed = W_fixed.assign_coords(y=y)
W_fixed.name = "vovecrtz"
W_fixed = W_fixed.to_dataset()
W_fixed

In [ ]:
W_fixed = W_fixed.astype('float32')
W_fixed.to_netcdf('W_1993-01fc.nc')#outpath